In [5]:
using JLD2
using Random
using Profile
using PProf
# using Revise 
using MPSTime
using Distributed
using Plots
using BenchmarkTools
using LaTeXStrings
using StatsBase

In [2]:
addprocs(1; env=["OMP_NUM_THREADS"=>"1", "JULIA_NUM_THREADS"=>"1"], enable_threaded_blas=false)
@everywhere using MPSTime, Distributed, Optimization, OptimizationBBO, BenchmarkTools, JLD2

In [3]:
@everywhere @load "../test/Data/ecg200/datasets/ecg200.jld2" X_test X_train y_test y_train
@everywhere opts = MPSOptions(verbosity=-1, log_level=0)


In [4]:
mps, info, test_states = fitMPS(X_train, y_train, X_test, y_test, opts);

In [4]:
pmap(x-> (fitMPS(X_train, y_train, X_test, y_test, opts)), [1]);
;

In [16]:
b = pmap(x-> (@benchmark fitMPS(X_train, y_train, X_test, y_test, opts)), [1]);
;


In [17]:
b[1]

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 12.091 s (9.98% GC) to evaluate,
 with a memory estimate of 7.86 GiB, over 32963986 allocations.

In [60]:
b = pmap(x-> (@benchmark fitMPS(X_train, y_train, X_test, y_test, opts) samples=3 evals=1 seconds=100), [1]);


In [61]:
b[1]
# b[1].memory

BenchmarkTools.Trial: 3 samples with 1 evaluation per sample.
 Range (min … max):  3.927 s …    4.374 s  ┊ GC (min … max): 5.85% … 15.96%
 Time  (median):     3.982 s               ┊ GC (median):    7.02%
 Time  (mean ± σ):   4.094 s ± 243.819 ms  ┊ GC (mean ± σ):  9.83% ±  5.53%

  █      █                                                 █  
  █▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  3.93 s         Histogram: frequency by time         4.37 s <

 Memory estimate: 2.50 GiB, allocs estimate: 10014069.

In [5]:
chis = 10:5:30#40 
nbs = length(chis)
nsweeps = 3

times = []
stds = []
mems = []

for chi in chis
    opts = MPSOptions(verbosity=-1, log_level=0, nsweeps=nsweeps, chi_max=chi, d=10)
    @everywhere opts=$opts
    b = pmap(x-> (@benchmark fitMPS(X_train, y_train, X_test, y_test, opts) samples=3 evals=1 seconds=60), [1]);

    t = mean(b[1].times) / 1e9
    s = std(b[1].times) / 1e9
    m = b[1].memory / 2^30 #( in MiB)
    push!(times, t)
    push!(stds, s)
    push!(mems, m)

    println("chi: $chi, trials: $(length(b[1].times))")
    println("t: $(t)s ", "pm",string(1.96 * s))
    println("m: $(m)MiB")

end



chi: 10, trials: 3
t: 4.050943773666667s pm0.5016267433937391
m: 2.499015137553215MiB
chi: 15, trials: 3
t: 7.527886534333333s pm0.5549147783814103
m: 4.311667695641518MiB
chi: 20, trials: 3
t: 11.879543933s pm0.25368685295562693
m: 6.705138549208641MiB
chi: 25, trials: 3
t: 19.67113332266667s pm0.7392081627189203
m: 9.605719909071922MiB
chi: 30, trials: 2
t: 35.844134002s pm7.802163435319291
m: 13.109912768006325MiB


In [6]:
@load "t_new.jld2" chis times stds mems
times_new = copy(times)
stds_new = copy(stds)
mems_new = copy(mems)

5-element Vector{Any}:
  2.499015137553215
  4.311667695641518
  6.705138549208641
  9.605719909071922
 13.109912768006325

In [7]:
@load "t_old.jld2" chis times stds mems


4-element Vector{Symbol}:
 :chis
 :times
 :stds
 :mems

In [8]:
mems

5-element Vector{Any}:
  7.233952149748802
 13.437170401215553
 21.876594930887222
 32.656362265348434
 45.6726955473423

In [10]:
p1 = scatter(chis, times, yerror=stds*1.96/3; label="ITensor")
scatter!(chis, times_new, yerror=stds_new*1.96/3; label="Optimised")

ylabel!("Mean runtime (s)")
xlabel!(L"\chi_{max}")

p2 = scatter(chis, mems, label="ITensor")
p2 = scatter!(chis, mems_new, label="Optimised")

ylabel!("Memory Allocations (GiB)")
xlabel!(L"\chi_{max}")

plot(p1,p2, size=(1000,600))
title!("ECG200 performance, 3 sweeps, d=20")
savefig("perf.png")

"/home/noodles/.julia/dev/MPSTime/Folds/perf.png"

In [21]:
println("t: $(5)s ", L"\pm",string(1.96 * 2))


t: 5s $\pm$3.92


In [24]:
b.memory / 1024

1.1171875